# Phase 2 — Image → Video with Wan 2.2 on a free Kaggle GPU

**Goal:** photo + motion prompt → stable 5-second MP4.

**Model: Wan 2.2 TI2V 5B.** One model that does both text→video and image→video. At 10 GB it fits the free T4 GPU at full quality, while the 14B image-to-video models would be far too slow on a T4.

### Before you run
1. Top menu → **Settings → Accelerator → GPU T4 x2**
2. Top menu → **Settings → Internet → On**
3. **Add your photo:** right panel → **Input → Upload** → drop your photo (e.g. `bride.jpg`) → give the dataset any name → **Create**
4. Run the cells **top to bottom** (or *Run All*)

First run: about 5–10 minutes of setup (install + ~18 GB download), then the generation itself.

## Step 0 — Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("❌ No GPU. Top menu → Settings → Accelerator → GPU T4 x2, then run again.")
elif "P100" in torch.cuda.get_device_name(0):
    print("⚠️ P100 selected. Switch to GPU T4 x2 — newer PyTorch versions may not support the P100.")
else:
    print("✅", torch.cuda.get_device_name(0))

## Step 1 — Install ComfyUI

In [ ]:
import os, re, shutil, subprocess, sys

# Big files (ComfyUI + ~10 GB of models) go to scratch disk; finished videos go to /kaggle/working
def pick_workdir(min_free_gb=25):
    for d in ["/kaggle/temp", "/tmp", "/kaggle/working"]:
        if os.path.isdir(d) and shutil.disk_usage(d).free / 1e9 >= min_free_gb:
            return d
    raise RuntimeError("Not enough free disk space for the models (need ~25 GB)")

WORK = pick_workdir()
COMFY = f"{WORK}/ComfyUI"
OUT = "/kaggle/working/outputs"
LOG = f"{WORK}/comfyui.log"
os.makedirs(OUT, exist_ok=True)
print("Working folder:", WORK, f"({shutil.disk_usage(WORK).free / 1e9:.0f} GB free)")

if not os.path.isdir(COMFY):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI "{COMFY}"

# Install ComfyUI's requirements, but keep Kaggle's own PyTorch (it already matches the GPU drivers)
reqs = open(f"{COMFY}/requirements.txt").read().splitlines()
reqs = [r for r in reqs if not re.match(r"^(torch|torchvision|torchaudio)\s*([<>=~!].*)?$", r.strip())]
open(f"{WORK}/comfy_requirements.txt", "w").write("\n".join(reqs))
!pip install -q -r "{WORK}/comfy_requirements.txt"
print("✅ ComfyUI installed")

## Step 2 — Download Wan 2.2 TI2V 5B (≈18 GB, a few minutes)

| File | What it does |
|---|---|
| `wan2.2_ti2v_5B_fp16` | The video model (5 billion parameters), text→video and image→video |
| `wan2.2_vae` | Wan 2.2's VAE. It compresses video more than Phase 1's VAE, so the model has less to compute per frame |
| `umt5_xxl_fp8_e4m3fn_scaled` | The same text encoder as Phase 1 |

In [ ]:
HF21 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"
HF22 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
MODELS = [
    # (repo, folder inside ComfyUI/models, file name, size)
    (HF22, "diffusion_models", "wan2.2_ti2v_5B_fp16.safetensors", "~10 GB"),
    (HF22, "vae", "wan2.2_vae.safetensors", "~1.4 GB"),
    (HF21, "text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "~6.7 GB"),
]
for repo, folder, name, size in MODELS:
    dest = f"{COMFY}/models/{folder}/{name}"
    if os.path.exists(dest):
        print("already downloaded:", name)
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"downloading {name} ({size}) ...")
    subprocess.run(["wget", "-q", "-c", "-O", dest + ".part", f"{repo}/{folder}/{name}"], check=True)
    os.rename(dest + ".part", dest)
print("✅ Models ready")

## Step 3 — Start ComfyUI in the background

In [ ]:
import json, time, urllib.request, urllib.error

API = "http://127.0.0.1:8188"

def comfy_ready():
    try:
        urllib.request.urlopen(f"{API}/system_stats", timeout=5)
        return True
    except Exception:
        return False

if not comfy_ready():
    server = subprocess.Popen(
        [sys.executable, "main.py", "--port", "8188"],
        cwd=COMFY, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
    )
    for _ in range(150):  # wait up to ~5 minutes
        if comfy_ready() or server.poll() is not None:
            break
        time.sleep(2)

if comfy_ready():
    print("✅ ComfyUI is running (no web page needed — we use its API)")
else:
    print("❌ ComfyUI did not start. Last lines of its log:\n")
    print("".join(open(LOG).readlines()[-30:]))

## Step 4 — Your settings

Start with the **quick 2-second test** (`FRAMES = 49`). When that works, set `FRAMES = 121` for the full 5 seconds and run Steps 4–6 again.

**Motion prompt tips:** describe what *moves* and how the *camera* moves, not what's in the photo (the model can already see it). Small, natural motions look best.

In [ ]:
IMAGE = ""     # "" = the first photo found in your uploaded Input. Or a file path, or an https:// link
PROMPT = ("The bride slowly turns toward the camera while her dupatta moves naturally in the wind, "
          "gentle smile, soft cinematic lighting, slow camera push-in, realistic")
NEGATIVE = ("blurry, low quality, distorted face, deformed hands, extra fingers, static image, "
            "watermark, text, subtitles, overexposed, flickering")
SEED = 42      # same seed + same settings = same video; change it for a different take
FRAMES = 49    # Wan 2.2 makes 24 frames per second: 49 ≈ 2 s (quick test), 121 = 5 s. Must be 4n+1
STEPS = 20     # more steps = cleaner but slower
HD = False     # False = 480p (faster). True = 704p, the model's native quality, but several times slower

## Step 5 — Validate and prepare the photo

The video size follows your photo: portrait photos make a vertical (reel) video, landscape photos a horizontal one. The photo is center-cropped to that shape, so check the preview: what you see is the first frame of the video.

In [ ]:
import glob, requests
from PIL import Image, ImageOps
from IPython.display import display

# 1. Find the photo
if IMAGE == "":
    photos = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True)
                    if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
    if not photos:
        raise RuntimeError("No photo found. Right panel → Input → Upload → add your photo, then run this cell again.")
    source = photos[0]
elif IMAGE.startswith("http"):
    source = f"{WORK}/downloaded_photo"
    urllib.request.urlretrieve(IMAGE, source)
else:
    source = IMAGE
print("Photo:", source)

# 2. Validate it
try:
    img = ImageOps.exif_transpose(Image.open(source)).convert("RGB")  # exif_transpose fixes phone-photo rotation
except Exception as e:
    raise RuntimeError(f"This file is not a usable image: {e}") from None
if min(img.size) < 400:
    print(f"⚠️ The photo is small ({img.width}×{img.height}); the video will look soft.")

# 3. Pick the video size from the photo's shape, then center-crop and resize to exactly that size
portrait = img.height > img.width
if HD:
    WIDTH, HEIGHT = (704, 1280) if portrait else (1280, 704)
else:
    WIDTH, HEIGHT = (480, 832) if portrait else (832, 480)
prepared = ImageOps.fit(img, (WIDTH, HEIGHT), Image.Resampling.LANCZOS)
name = f"i2v_input_{int(time.time())}.png"
prepared.save(f"{WORK}/{name}")
print(f"Video size: {WIDTH}×{HEIGHT} ({'portrait' if portrait else 'landscape'}). First frame:")
display(prepared.resize((WIDTH // 2, HEIGHT // 2)))

# 4. Upload it to ComfyUI (the same API call the backend will make in Phase 3)
with open(f"{WORK}/{name}", "rb") as f:
    response = requests.post(f"{API}/upload/image", files={"image": (name, f, "image/png")},
                             data={"overwrite": "true"}, timeout=60)
response.raise_for_status()
UPLOADED = response.json()["name"]
print("✅ Uploaded to ComfyUI as", UPLOADED)

## Step 6 — Generate

Same flow as Phase 1: send the job, wait for the frames, join them into an MP4 with FFmpeg.
On a free T4 this takes **minutes, not seconds**: expect roughly 5 minutes for the 2-second test and 15+ minutes for 5 seconds at 480p.

In [ ]:
import csv
from IPython.display import Video, display

def build_workflow(prefix):
    """Image → video workflow in ComfyUI API format. Compared with Phase 1, the empty
    latent is replaced by one that starts from your photo (nodes 7 and 8)."""
    return {
        "1": {"class_type": "UNETLoader",  # loads Wan 2.2 TI2V 5B
              "inputs": {"unet_name": "wan2.2_ti2v_5B_fp16.safetensors", "weight_dtype": "default"}},
        "2": {"class_type": "ModelSamplingSD3",  # noise-schedule shift that Wan expects
              "inputs": {"model": ["1", 0], "shift": 8.0}},
        "3": {"class_type": "CLIPLoader",  # loads the text encoder
              "inputs": {"clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan"}},
        "4": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["3", 0], "text": PROMPT}},
        "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["3", 0], "text": NEGATIVE}},
        "6": {"class_type": "VAELoader", "inputs": {"vae_name": "wan2.2_vae.safetensors"}},
        "7": {"class_type": "LoadImage", "inputs": {"image": UPLOADED}},  # your prepared photo
        "8": {"class_type": "Wan22ImageToVideoLatent",  # encodes the photo as the video's first frame
              "inputs": {"vae": ["6", 0], "start_image": ["7", 0], "width": WIDTH, "height": HEIGHT,
                         "length": FRAMES, "batch_size": 1}},
        "9": {"class_type": "KSampler",  # the diffusion loop: removes noise step by step
              "inputs": {"model": ["2", 0], "positive": ["4", 0], "negative": ["5", 0],
                         "latent_image": ["8", 0], "seed": SEED, "steps": STEPS, "cfg": 5.0,
                         "sampler_name": "uni_pc", "scheduler": "simple", "denoise": 1.0}},
        "10": {"class_type": "VAEDecode",  # latents → real frames
               "inputs": {"samples": ["9", 0], "vae": ["6", 0]}},
        "11": {"class_type": "SaveImage",  # saves every frame as a PNG
               "inputs": {"images": ["10", 0], "filename_prefix": prefix}},
    }

def comfy_alive():
    """True while the ComfyUI process is running. A busy ComfyUI can be too slow
    to answer HTTP requests, which doesn't mean it stopped."""
    proc = globals().get("server")
    return proc.poll() is None if proc is not None else comfy_ready()

def get_json(path):
    return json.load(urllib.request.urlopen(f"{API}{path}", timeout=60))

def still_queued(pid):
    try:
        queue = get_json("/queue")
    except Exception:
        return False
    return any(item[1] == pid for item in queue["queue_running"] + queue["queue_pending"])

if not comfy_alive():
    raise RuntimeError("ComfyUI is not running: run Step 3 again. Last log lines:\n"
                       + "".join(open(LOG).readlines()[-30:]))

# 1. Send the job (like POST /generate/image in the plan), unless the last one is still running
if globals().get("prompt_id") and still_queued(prompt_id):
    print("The last job is still running, waiting for it:", prompt_id)
    job_start = globals().get("job_start", time.time())
else:
    prefix = f"wan_i2v_{int(time.time())}"
    request = urllib.request.Request(
        f"{API}/prompt",
        data=json.dumps({"prompt": build_workflow(prefix)}).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        prompt_id = json.load(urllib.request.urlopen(request))["prompt_id"]
    except urllib.error.HTTPError as e:
        raise RuntimeError("ComfyUI rejected the workflow:\n" + e.read().decode()) from None
    job_start = time.time()

# 2. Poll until it's finished (like GET /jobs/{id} in the plan)
log_start = os.path.getsize(LOG)
while True:
    try:
        history = get_json(f"/history/{prompt_id}")
        if prompt_id in history:
            break
    except Exception:
        pass  # ComfyUI can be slow to answer while the GPU is busy; ask again next time
    if not comfy_alive():
        raise RuntimeError("ComfyUI stopped. Last log lines:\n" + "".join(open(LOG).readlines()[-30:]))
    with open(LOG) as f:
        f.seek(log_start)
        bars = [l.strip() for l in f.read().replace("\r", "\n").splitlines() if "%|" in l]
    status = bars[-1][:90] if bars else "loading models / encoding prompt and photo..."
    print(f"\r{int(time.time() - job_start):>5}s  {status}", end="", flush=True)
    time.sleep(10)

result = history[prompt_id]
if result.get("status", {}).get("status_str") == "error":
    raise RuntimeError("Generation failed:\n" + json.dumps(result["status"].get("messages"), indent=1)[-3000:])
frames = sorted(img["filename"] for img in result["outputs"]["11"]["images"])
minutes = (time.time() - job_start) / 60
print(f"\n✅ Generated {len(frames)} frames in {minutes:.1f} min")

# 3. Frames → MP4 with FFmpeg (24 fps is Wan 2.2's native frame rate)
ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    !pip install -q imageio-ffmpeg
    import imageio_ffmpeg
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
first = int(re.search(r"_(\d+)_\.png$", frames[0]).group(1))
mp4 = f"{OUT}/{prefix}.mp4"
subprocess.run([ffmpeg, "-y", "-loglevel", "error", "-framerate", "24", "-start_number", str(first),
                "-i", f"{COMFY}/output/{prefix}_%05d_.png",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", mp4], check=True)

# 4. Record the run
runs_csv = f"{OUT}/runs.csv"
is_new = not os.path.exists(runs_csv)
with open(runs_csv, "a", newline="") as f:
    writer = csv.writer(f)
    if is_new:
        writer.writerow(["file", "photo", "prompt", "seed", "width", "height", "frames", "steps", "minutes"])
    writer.writerow([f"{prefix}.mp4", os.path.basename(source), PROMPT, SEED, WIDTH, HEIGHT,
                     FRAMES, STEPS, f"{minutes:.1f}"])

display(Video(mp4, embed=True, width=480 if HEIGHT > WIDTH else 640))
print("Saved:", mp4, "→ download it from the Output section in the right panel")

## Next

- **Full 5 seconds:** set `FRAMES = 121` in Step 4, then run Steps 4, 5 and 6 again.
- **New photo:** upload it to the same Input (or set `IMAGE` to its path), then run Steps 4–6.
- **Experiment:** change the motion prompt and `SEED`; try `HD = True` once to see the quality vs. time trade-off. Every run is logged in `outputs/runs.csv`.
- **When you're done:** stop the session (power icon, top right) so you don't waste your weekly GPU hours.
- **If a cell fails, or the video is black or pure noise:** copy the message and ask for help.

### Phase 2 is done when
- You have a **stable 5-second clip** from a photo: the person stays the same person, no melting or flicker.
- You've tried a few motion prompts and know which kinds of motion work well.

### Phase 2 learning checklist
| Part of the workflow | Concept to learn |
|---|---|
| Step 5 (crop, resize, upload) | Image preprocessing, aspect ratios, why sizes must be multiples of 32 |
| `Wan22ImageToVideoLatent` | Image conditioning: the photo becomes the first latent frame, and a mask tells the model to keep it |
| `wan2.2_vae` vs Phase 1's VAE | VAE compression ratio and why it changes speed |
| Motion prompt | Prompting for motion and camera instead of describing the scene |
| `HD` on vs off | Resolution vs. generation time (check `runs.csv`) |